# 🎬 Montage automatique Locapay

Chargez une **vidéo brute face caméra** : le notebook la **transcrit**, **coupe les blancs et les hésitations**, ajoute les **sous-titres animés mot par mot** (italique fin + mot clé en gras capitales), les **recadrages**, les **écrans de couleur Locapay**, le **logo**, des **effets sonores légers** et, si vous voulez, une **musique de fond** qui baisse quand on parle. Vous récupérez un MP4 vertical 1080×1920 prêt à publier.

**Mode d'emploi**
1. Vérifiez que le GPU est activé : menu **Exécution › Modifier le type d'exécution › T4 GPU** (gratuit).
2. Menu **Exécution › Tout exécuter**.
3. À l'étape 3, cliquez sur **Choisir des fichiers** et sélectionnez votre vidéo (et votre musique si vous l'avez cochée).
4. Le téléchargement de la vidéo montée démarre tout seul à la fin.

💡 Si un mot est mal transcrit (ex. « Locapé »), corrigez-le à l'étape 5 puis relancez seulement les étapes 5 et 6 (bouton ▶ à gauche de chaque étape).

In [ ]:
#@title 1️⃣ Installation (1 à 2 minutes)
import os, subprocess, base64, gzip, sys
os.makedirs("/content/montage/polices", exist_ok=True)
subprocess.run("pip -q install -U openai-whisper", shell=True, check=True)
P = "/content/montage/polices"
for nom, url in [("serif.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/playfairdisplay/PlayfairDisplay-Italic%5Bwght%5D.ttf"),
                 ("gras.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat%5Bwght%5D.ttf")]:
    subprocess.run(["wget", "-q", "-O", f"{P}/{nom}", url])
open("/content/montage/montage_auto.py", "wb").write(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/7193ZIbR5befT9FbckTXRDRINBNtkhwMGtKoih6KYnB5ow2DMHYAlDoLhJ/qip0N9nuiHkDX6+vJhwO72g2Yr0XDtsb4yv1m+wL+BX8fedkVmX9dLMpacyQGkBV/pw8efLkyfOXvu/vfLVeZeFx5IXbbL0Ms/j7beT9+I9emr1dRN7z9TTchG+9H/+8s3ebfzu/i2dXP6y9SbLNIm8eTiNvGi6vfkhCz9v7jedlSbhKp0m8yeL1ygu+PYnTTZS09CVqTtfbTeTNotSbLMLVNN3h83S9TfeyOEvwOFzFaC71luvM24SJfAZxFi4E7nm88u7Is+ni6gcvWnnHSZgChA2LRGlL2kuiaThLMGY0t8iiZMUG73iocRwl3rv1eulNgZR4tVW4ABbBBlh4vl1E24Rfl2GCLqXBxfp4rUWj+TzKUkC8WhNabTLVd8ttKkCi7ny9mnnBWrCwihaLqO3NttM38epYIfzqxT3vNEqyeBouvF73Qfe893C/622Sqz9l3tUfvM12sogBa3CngtFOdp55UeZ10iRrdXZ8zG+83KyTzHsNmNoAOjtpe+u0DSS0vfRkm8ULfG4nm2Q9jVI8PwtPo515AhzMwiycLsI0xUBMI/mjNlAdLWY7tvXVdrl56wHVq41WfvHsua30bAlUt/Xj8yQ8M1+/iIl7+wP43tn5tu192fa+eHHkDWTUbY/DbnsH3Z2jl3h270G3293Z2fnIG/yMf16CWVlw+nf+bT6gHfnrvYz0TX/Hwz8z2+NeH8shAQD+R/tP9j+73/M9599HoNVoCxTEWb5eAkszd0iTrVJr+0Vr3cef7T++V2mNE28bkoqbdRpzdsdcCGNdCH1vvliHGVrpdg4PtWIXv06wjIE2fJuEqdReTcaRwDI2APS9eMWK+/JaKXZsKDYHTSnXd8DKH623nh9up9uVLw2crhfbZTQ21O3C1XsgBdLtZpPEyygZn0QYSMihoKPJer1AsVcJloR0EG1BmifbZRugn3Q6HR17uE3ReHjuNnzvvlf+9xG4194sWlz9oe2Bawnv8KI0I843YARBqlOANXscjcPTcJWVAO26bzcV9Pbuy9vZNomi8QZNl8HZ79y7BpLtqmA1Ak28SsEKr36Q9shnilZ6tpsIjApMAVVKKJJ35DPjaIWJjFf1twrgnK+KVnUK5vH0BPxizAbyKfYNcS3iaTROoySeN78iC02jyruPPHB7MKT1PF6Qo8l2IW+yMAZDG2fReRa5kHQrM7bcLrJ4gx7CzHDURWgqywbgcH2dnTXIFCwf8HCTsFR8r0oJq/VykpA9nwuXRi0vnBMB5PLRKo2WEwMp3o2ni6hGitUWMY/ulvL05eMj77PHL569evz8yZHsQgqSrvISsrColth4/HqjUk4GihXLvUIg7XumhncXI3i9TadYsil/YDfSH9ILt0FZSHlPm/WmoZdwA/CEeUhXpg+URZPchLb4lLWswMssjl0SNK1PwVdmfoHjMAOxLVBZXvATDBx0kgFFaDlZx3yIDT1lSUPxwoaUQKu0q+SEud+iQ+UQXvDpXrJeLFoAeHjhz6KJ3/Z81Pa9IIuWG6GQZBq15KnQNwtkbzeRD+LgrvIfT+NZtL4cSRelxvveIk5JPrKNBbNoHoIcxxBYsnXydsCXLWw0TzDF33z99eOnT7glXUg7vhk8OvGj7wd24ANQ+EFfhitdyG+/rVUUffUKh5UKD/rH4XIZDrqdhw/a0/VinUxC8rJokKR4iA4m6WBPviRLPtjvT5byYD/vKp+Ien+9++X+evtFf4e2AZm8Blgrdbs1+PYsgPi8R7hcsApSaGj6gdt0t/MJa10C/ZgW7yQ630+OJ8FJS/fkE25yHUxkvAlanYV+8T/ylcEnEdpZedl2s4gCMIjgZBj3Y+zC+yPsioctEH3ixWAdXgC5Yr/t3Wu1fr5AUZLBBOqxcKtZtNnG2Fuj42W0ytLAfjFDkdU48IZKn4QMBQibLafF7Msz86pzHGWBf7ZOZikIfjhqeXg5HBWFhQujYeAmOBtKSX/UsjgrFYvnXlauKICR2P0zzBRkCT/1DRtnY2kWJhlaw/PIfR6t2MdlrSl0cKYAY6+YhJN4EWdv/ZYXQ1QET/16vYrq/QsMQ3/jjwAIWOtqFuQduc0QjINWHXzgtQPOB5iCZYku+MbQFb5SGCtNXWAmBmLzcnHuBV+FU+/xBqTkHaFDUCz6m4Nmo2TvTE8uWubZKhNm+OIzFFij3zAuCny2XoSTVsc3G6cRivVjEU86KoJDrsgwycsdgzT7u5O+RX/LoAUK9fzPw+SMDDDE8SEvsQzB/FaRKRImy8N7WqLcRwesczYGTNOAoxsbAP1WgX+DJb72LSDXN6KYuKkdLXGLpm5oQ9GpbSRhjD3z5RZns2X0JEnWSeA/5u5lZpMbfvk8BIkLmxk37gCDspNSm0WItOVpw3yRL3z1zedPsMePv3r+t7IkANJkTQbGxqbr5XK7Ah3eNZX2FhQg904P9rQc1w728sX1FeT1HtHdrlKxv4xm8XZ5fV19r5UhnJuubwEca1zmQ/vi8dGrJy/Lo7thHPqlXQJPv1WAyL9bRm4nJokC2ZZxGF1DUI4Gvu0FO8nxFr/n3MhPcQCabBchyg98v1iYr0rzaxQIHg6tU+9knax5oDuOyuoBQxzTk3WcxuCgEG4oa5p1UyxNU25wDW+wZGyLDXShFCRrlrazuPJXOABhM5pXoCdNtr0LxcNl619//98w8DnElJMBJaOCswlQRbMdC9kkx+UGB/vxGkes+TiJNuuBQ7nCfrUPi/OWQTZQNVCstxuZcOUfd5Ix1x4WFQQwAbKNNQYBM1yMwZmXm2zgTBw3JnL4WzUO9jrTYy7+wxnsNJbTLo4Rgy/CRepiQ1lD8x6bYI8y37E/NMyZYUrO3kp9RZmZ2ak05PUVkfa+uSyzlFtNK/dZt4ugvCibJ24GzEDg8qmvQ7tY6Bvo2saUewc+IHvgFx0AE1DVjEk9DTRTJYG/3Pz+QnNb6gZnApWB+vYrvp11+IUMS+QUPpBvFFdWWgCf+OWKEXzq/L68djS5FBak0k+qkldrdKnSmxHQUkt3SkQ4UUxP3Acuc8BsyvlqOwt9kcZYuMOfHYwcmooY0sOCm3sEPKHgZpvvp1J3oM9kw1caEQBzpqqMuMSdC+7t7LdK1f6//uc//N///Z8870Uo+s6nL36L4xfavfojGt4V7r/rcfuOU26qG1A1NC4rUTouoZZIqaT8YRqnrY7nVxDpP55m8Wn0jgdqbfjJOcpuZf38xsMSiOfUaqInUrM3243c96/usVbHIW8z3oHdlnbc5fnZCcdLwoFmJB+C3TCg4bYLFMpt6KgS7wL4vORiNboitGv5LYTP2ViK5+vRLEN8GAnzZy2yZpK7xcrz5pve4SDISYF01LqmufetwjZVj5N1GrmL8kOY7c7Ol0+OoBZ59eybr+XA7EOnR3I7ibb8wC/9qeLCyVI+lkt5qH+h+pMPUzE6gfxw9OqbFzzO8BTh+wtRFFErsiAhQXUhqg0QK6Z5RjqENLeEpOjN1tACAlGrGNqWxHsNstpCDeBR4e6BpFYYPuQM/IkXqTxNd5YoBG0WJYgYizH2Flv8z72Df1LvLTVAtGqgH/yf8VvqXf1DiC68qfc9SlPH//0W/6E2QAA8V/9CmEJq7kN8bs93qA1M+S4FPj3Rdi2XbOhPGTVCMVXp4Sm1ICEXDEtinBs8lRWXJVd/5N+If6DM2UATIvKOyEEpFc+kaKrQdmb8NT1BK6c0RSwx8Az/o3dgCQo3FPfYNTumqh/HMtDXqfxdrYkgPI+5zwA3UAwl1FileDKJIza1zXYy4pDfIv2b4tQOJrQGMlbo5nQdQx8qH5g4fMNHPI05WPSBYcYJ9l9oz+dC2tNdwc4uZur7rfnLydl5vQvErOTtbBdTzz8ghl10gbEvY0yNNgj1LjRfgiPR5npKtx5PhHwXraZQdlM99RrQzKLXoXeKswXpeAc6NwxjefUnjHYJ9oHeyVpl+KhKTMvMbqJtxj/nQKv+AVQx/08iUGknhWozC6Bd+OLxs0+fi/7ogwmY9GIohmQ3s/PaMN0F4e2gvRWJ/fX7yb1OydMPIpAdSyBvuWRAXMAZ10/mUqhQbE5B7kzlVFlCmB4XqAcNMrNBGSaURB2YqoLEH/6H786u/nD1X67+69UPV3+8+tPVP17909V/v/ofV//z6l+u/nz1f67+Yfdff//3eyOyEPyfgYGfRQka70A4XsAqiY3u93/Pt7u+7RAKAzEiJlCvJ9Dbi56QZ4eUQlb+pDiJOA+xle0uaLG5+mFgLDePSF6wKGCM22TgfE937Xkj485ONnmZK4A2sgJiHaogBIN99N1q5JeAIFfz3d0b0oA/8FlRWiirVEJYVNCNvDGtojD0YTVdUGiVRHWdjAA7lFnJS7Won5mUFEvxXPQ6Urp2mBf1C3+DSZQVX0uCztdFnTcoId1BE3RmRXnTxRsWr/ShFovVlC1jnNQuQCYg/tqdv/rrR33s7KM7/wbDrrZnN/tZPIW4jQ16oGN9M4LeMJAmh92RKGWkeRXE/FbRAobTrG8ic1Tq4qkMcmicBGIYofwwoREVNllplWrxcNFw1KWKJyKfkHpkA6I2NGdd2jQgb0nTyhuGtuERpFQ96srutab0sU2K0y7YwWlEzjLQpu0CdKfRlqnNpJk6ixMDvQihrEc6i0jvZfQT+7bJ4V5v5JCZ+9i7Myg1a4xR8Uy0pZA2g7MWVbqFPG5rK1TZGjqfNnWXSmfpdhlI9ZZBu0N4sEZvpIl3IGHbTFt7c8CboZmA3gV7bKHlfcxad7WnJjJQ7emZ0Z6qCjOjstLoTc0TOiuE5wEU5fcBGlqFIn6/ZcptUA7K8cuCzjJiZnYNgQnGgBkFevO+VXBWItSNpXBL3aZV8pwYVt+b+OBjwzhlJ6PlJFJPCYc7SlGYC3H6ufpB3EuWa9lOIXoDGbDmkCnLnijUvfrxn6Glx/b32gvY6no63aI5GBrs8QIrSPU5kW6LLLU5gUEhanmP6t2JMM7esLXiJ6SU12uKc5X+ceCgvJDywPF3Ikf8HQFgLe5isYiAhLDFfYyyjlqpf7vKG5Fhsbvc7l0MK+DGC0srxTm1TFOW4o5LpVVJESWkrhypVeaQSrvTGCwqklJTeT/l+8oWMZQ5nRr9O7ARRb5qiRX9frEOpHrKIrPAtN323kQwhYXLyQyibd/b47lqOvRjME9362ljlgae87Jtf7yusG2yhqDr/Rql+ee192ueHYW2WpUtxzjgRIUqYxOuVDWXwqTzGiunN9pxNf75U5QquK0ZvIiBIg0AOjYFesfZxR+ZX+A6sGHwp1lHxVPuFa0ynmpoLm3EN+D7PWMUxHNKM+krk5O87IHnSgTnMktZMSySqA6rlQtQokYocEOsEcvSeG06Vu8BaU13kjeyWbe9bv4YB2NA+sadwH5VmHiD2VhxnvMisj+oDCEzxhLnBsXF+LDrHUfBisLFQFFSl0ZcanjTl55GtULk/nf+UsTg/nvDblalx2Sj/SaIckFBQB+1HmntXomWMTRy9mbLlezWFeW0MwPWcOL5nddQyBjxSZSgOvyUFCOKSMPsh9wU/DKX+QWsoqdwypuFue8BDq+pOF0dPfnts+fjz775+otnj7/+7Im6MNld7HWUUeyWMZYHdPOiKIkwugSE2jKhchRwNBMjK/FDuHib5ptbGm3jxaACXbHNfW69E+mmUlL9QibTrSfl4S2Nj0UWmuoJiwdfOP2oZivfCBUXgIvq2wSiW4CVN49pSs9VVDgNkx306UwZYxWKn4gxkpjD1EJcCTF9zqaX0EZGmfDqB9n9Apo65DirQIonUgL9H/ZpPeyao3Kfh0SomuBuJjv4cZjMqI0L8SNZxVd/TJy9dU7JcaaSpxfgy4Tn37yPcALkw4cEOkI5eWMJJtQJ2h3O4AubuBhcyFwcoQxbihBiBLfCCG4BUZXDkMvCka0s+yeiSpV1hZdcr6MqS+JiwDoXNEWnssihWAT5kY2XhFYIS1Zi1aK1I0O+phOeKCJpNe+8O1KOgtZ7+w2nKTN+ywxQBYzA4qLYD7aJLRHncnlp4A3t7DSfsN5gV56dl7FqajvwnYlEodvpqHCWQM2RoyleT6TYmZrocykclR1zf9nI7wwKhG6FcrbUghzNHcL8oBAqHQhvYtXCUCMLIjWVe3pgljOvVBDx9deYzYP7N7ZZ7GJYpkkWpWWkE8UE0QFeOImsULQvbIKzbsH5zUD7rBy6tW07M76z1GU9i0bQXdF/3fJbzUeJFQ4C3OJ62LLoBoWDRVpsWeISJY+KfcvHhNEJanberAr2hV3STGt3iTPdJZy55BGEQ0YpfuCXGRIemG/XNK6Mw+8L/mxRmmTKjy4NqX6kPCt1mBY93lwOQ+ZE9IX0AACXEr0XcX+Qc14oUikMGf84dWCO53N4G1izy1GEQwoY7FdhBkVM4q6MsLwugH1nNqEdHOQbUzg0uCtLtiS212EL5LH/HoGKPU480Y5mIgD1RHy5N6rJT5OhzDb5CnrlLJOd7N+vcxP4O4ZvSk9fTxyYJw0wixAVU+1RQUugZoXXUBq9htuKOGVVnIYA1TYbK0s3wqWM3XydqJz3ejLsm1ciyb0Oa9yYAMjqOZQatiE8OhCnJrej+qiBFEuRI7tY5r6WjuRYR31svjl7F5Ph7mp3dOlb2ddt/jYWamUfc7+09/XN3tbUUasJZrM6eEwRM8sNs1k65H/2DV0RYRz+3ePnzz4XSYaMy//6x3+mLPu5bMH89gUoBR+vRF9EexVkDyq0Wnz6mRUz+OPq94Vswt9PFTR8Oyqcb30rLi0g2ETpuBDrKvuHldOGmyFYFriQKjk2hoypb3SeCT3rs01OnxWG4stU4bXwoRFZsrBjMw8X2liv26U+ZrjLUrtwQvN+VXWnye2aPpaa7/34vyzb2zg0pGKj+8QqQwREO2/yA7xsNHI2IEWExdSY1gSIpyW88Nh23io8F1UjbcVTulOhM+K+yz+wm+iHUWHzQdSgq3ZmQ6XYXLLS6WpQ04jeH7oT8Y9e7KoxV52w+4WzftvKf2pNKuiBWjWjjFTtkPHuDaLzDgVjBLKIVDyHjAy7GNdiRsV4pGaL7SqXfrFi5gxKsQrMSBSYlDfpJyrmEFO0EBxrcs3mxAzVKvEqCDAm+oJ9aGGo86iqCuQXtYlDkVJASA9aw/6DQhww6lpFBg4J1rFSKg4PRkYV3zaTro8PR5UHn9QVIdrie3YLUzatOGnWNI1CItyZNyf5chNFI3+bpaYbsPCcy9Z7uy2fxMjA7RM2afeTOlh8AbBUdaVypON1y7qUTUbvAeBG5fSNCur/H+j5AF20q4+uqKJzWMpevrdTSOdK6XwEPNTYFq9RUP9kJbVVTfNlBTmOTvqjXBizMTBcwmJMgZYzSOnLgWCEqx/o0k/yF0kv03MyONPCRjfounYYqyC6vJ4deemDya60r/5MLQcDgjTQRJS4RmsOt1MI+YnGCxnOLPE5be9lP48yq/iBN6p7RXFjT6r58bUX7R3kJpiXneYQq6pKabisNy0q2bpiSA05hfHNbFv+Hk91JWVKa+QajMomPPXX/V242Ja8dcUIvF6IAT6DHlV4PXcZsP5TiR+1Tj7XmAYNPgZiPFmaM4/BzR0QavdBfrRgWMm79Urt66bbYL3yjConDzQFcvNQs5aJz1pFFff82+gltJp4Y9nJkkd6OsNKpScNHkNaftkpottKa6qxwsAM+5H7mv0X0m9cLM26/lFqFUsffEIa7hs49SBZPNJzpLQPB7t4dFnMyDthT2yu6OEdxKZ8SjADbT6xfRAJLztO8F3LrTfRepDGzAp55w77TlGTalsDxbYyM00gYTJYjlNhoAPO8UQxN7G0UjmwlQrocNxHbQMx1DX5Y50E8sF39scN84BqdhbelSkNb4oBgdbelWlNirXKMSHC3fJeyx0ZNjj0WVEG8y52lwVZE7QRTdHQbbM+yFfgPE/bhTgmwItD+J04HdGkbCJb307VbsGwIMwXQxvbtF3es78U1YZlacfvnT8eu3XiHonqqgm9cn4vjAa9thwiUbpqT1qKw1tbDP2KmNk5jt/gpCPyjvxJXWeYL+MQPOJlpxwGKvviJ/ctkelpfWlVf3hVP7ZORYINFCBlZHdMFaqt9uu6QqLLkswwpM40I/0RP1NjJWgspwCZPS9H+gUw2qdPiE9NEqzhPieIWh7O4jD2fqXmIv6ynq0gxoBeJK2K/pBtmg1AY2W7xf4t+y4LONrKob8WQsweqdywsSjbyDybMOlkQ0Azl1Y1wFKYeDUu1Lz6Gapk2biqW2Dfxjra04oKLLXjUJVcanppB8b3qHuvE8U/XNW9QsBhYsRNa30bvnYtba/Fjgk4mPggWgVNTjE3KMwPujWNednN4/1KcyEyAvAb5bI98opKkK9ogXQwv/EOsdiuLXiDat6wv78aGKD090/T5XcetH7q7L5fmd/QjlkCxvcTu2IsJ2gtKcdlnqcjkKKGusC3gTLNj39u080hyuw3NPDjn8VO1KdL4EaCjyGU5w2J8jRfNcdxwVM5TwayFvmlg4BjuPOQL5vXw+MYi6/4UbbRq8lYm2sRl7nOb3VOxWltTul91VP5tvqGGkKjzK/NfCG6cgKPZXEYwseYjDdmc6UbHZdqzTW1USpk912zUlfnSkrN9JfXtsXSggptg2xNDBzdChWuzjtwvYfWilGtxx3Ed4sjJV89LW+0x3GDHchO74fZgd5xRxZR1JgjdGBFAYh/9KHPbRXF6f48s+yrQi3F0DHv+sz4GOQkKNPeix4WMkC8MvIj2m27hhCROg5zka1YikC2DFCoUYjlzNj24ezQslaBszKWztIGLhMbT1VT36rzxGU9p2+3A9LuPYcriOBEtiZAtYY9Gbr+khaE+pmnoNVgEntaCPRGVD9LcyuR+LoZ85B4GvpohRs94xd8icbHL7Z3WTovNeS7CZZQEiLJiWZp0ENavtRZKskMdlb2DKBeQC87tfwjrcLFMEfRU8HLYUGAGepnTGPyVKjH6Gye5ocih5qIKk5mXNCpcq79tm17z9sX5a6kDvk1G41HhdpEf6mt5ddM6NEdlV1ySCurTWeBNbah8zBXYJukhR5WLR0JGZrMz5CduNbSuGJgLNyIpO1K3PaUok8XlDvFdhcABXv4WSY82LQSPZo6QxYs0IUX3n7hJA1i7tnKZx07aWzFtYr4o03exrI0iUQFyuWmlUpuYjEkS3QeuCi946JbRFyOaVrxoyFslpjZh7NalWxkwDYbwMtOnpsHS7b+dL9VOSO0FUvFaja+boKSCjdVcHV9UFq1AOC88CskEthxdUj5ulMdj/BA/pRPOnFSAqaikZ/4bZYMnjwV+ypkXTGw4rNss/BFdWffcS+0Zw/yO9kV83waZivsdFs22DZNpvtQb4mEjoVkBliSy4cGtrLzmgriqmE65GrIwH9VTC8jKXccFZEeeajKUrx5Lczq5+dXoHbPerkyJ00AS+WSHE3zxSBwCVlOROvFxFZQFZsBZ8lb57yMqcyTTSFSaxsxyqzSlONs69aV+hANsnHe03jydrwKlwj0sk8c7cv5NNpkCG/jB7O0lEmd+aYqeJzv3FzPFGuAX8ZbgK/zL+lPqEGx7uZzgxH1i040ImDOQ89ksj43Wn6TcYnc9EBPclAOM+FGgsldiJr7Y75n7T2e4OwDNc1I3h2D484qOgv855CbAraBRWqYWJ4GrMM/gVRqddh/ELDrPYKoX9jBPdS0Q8CwB3P6ti8Wg959057tVT7h70wVRuAkGOs8lWilcPXpYpsED+3msywD+vLpp48dWLl7mf9akOXjZQcxtuFicxIagJW6RIy5ZStNg0cD1w39mmEH+/fhKV76k4+oIwCO6SfBhGGRNO8uxnjZ1tlLtCuZRBCMpj57ISvL7AGy1saMPxyPGUaA/l+67vARPWqwboDrj6cnrUbWsJKBud7JJ+Qw67TDYHVo7GF/S4MpZa7pCXZ5NwgYJD0mw7ed+HePJA/G3efYc8Lk7V2ugfTuEfVT4nIdLu4+jdbJcRx6z5gNcNrJsrl/U2Cyf3ebJndTnC2ju3NpzS6pu4wIO93e/Rwfv9seMTPWntNoFc7JB8L5GOxi4X26Xsx+ORjByPfyFt3478W8sxHfRn6ZmC+JfZIQdqihnRxg7eK3Jv5q2+mwXybl5qdIo2CimHLSoS5FqUYySuXcqRzdE5RflgN8jCBddFHzj9EMlfSunvgNHsJ5RQbyDPKdI8eE3Tr8J+ewWhN5fo6VVsWXt9Tb9IN7S53evpLIa0OkeY9p6xbew7fv5PrWrTOC25T1WpCUZLQAvix7LmQdkK84K4jYoZnLxKs9z1DmqwiSB9rVSubpy2xJm4YHFiKmtNuAMQYQiV62Tdo5x3PhBbwB9pAJk/V5/LD5ED0TdcRzyklI1wbRgUg+M+jpoGfV2E2Tv01yoYVX/5QV3gRvxllusrjXNtmMXnbcFHbUCUGT7YQASIqmp47UhPVe9yE8LiRI47ZSnlCe07gIjq1Ou82vPKCN6o70jS3kzgzS+TAw86cHTU6iGHZxjDW6Zce3r6IubCY3jaPlKfkDWh72qyr04pQ7DIoAznbnUV8DkWibP6MrvLE0GloT/XT+ht3Um9Xg3g+FUJ3uGqAskLmQCgtxzBE0tPXQrCYocfgrV+eBM61rc6UV65yizdfxLF7tSnH5Gh6Li2AQMAfqA8pAweGDtvcJY8OC+/h2KN/u4dv9/VarOYNWQ5sapmNfQHbDAjDhOnkZfdjYYqx0HjjSZb5cRX+eZu4A5uItI/zZLHrTMbf8LG3d5HMnSOH2wZI59pq1tlG6MSPrdvYfYAB5P43FF2dGMx5TtbDvqreWKfXhbO9j1VTKk71qpKzr+HnGQ9K3Ygd6cNi/dkj1g3RONvbUy86gsThrC0MS4A4qwLUJW+XgDPNdInlI75UVCSdmlIkdYmI1IzpIqWaGaZ7WBvrW4PVLFKRYUM9Dy0ro6m7VcCVg2/EsTgTwvP86mqhFDr6lWAp3mrtNVjA2GOQS7BlF7JOWJQ701Ix64c0OgtvoaU+beMsz9eIEP9GQfchAtMaGzmmuOjtT6qiVeMu3aMug9Qbrr/U+G+a8qsai8h2hzktMelSyhofQaz6A/ukTcIjDbmPAtpR1l/x1a7u+rl1ZoLzEfYoVRmLTZWzX6o3rtLRGD/M1Wofll1mdt1yZ9VW5ODECQeP6KxU9b8v6KJEtEljLJx9dsyo+jIh/EQJuJl7QmbQuOi75ZgXBCD5XufvqudH2ESnnbdEyiW2/dLSkmjLgH5hyPv7YO8jDhzHHyZjZaiOqtE9DcAQZBMAHFtpFItuBZLA1fWooBwzKVQWIfZtX05yH4lXkiOXaVe1AzJ4lK+i52fTetni8d5rmetHhb/AinYpR+8G+GBJ63Fqieu+SeVetEp2umuWJL/GFIKn2Ooe5xhst0oDz8OFDJ/QdZLCyZNdTwHCaP4tnYKwfowrXWfkd0qscn2TmpZN/bp8zuUSSizR+FwWBttwyOdY7nz57/uzrJ49fFjWEMIrO9gCL0vEjZWlOX3tsSWn7joDRu8dlyDmPDAxlbmehyacMI+/VX+9DZ7t5G4iWZb9Qs/AH2A3keibIh+OY38L+w36NpvlUEkAHpwAibBkArp32/YZ5//lqyUqy/yB9u8qQZTpjaizwinGyOlbDQcJguGXHZhnG8+ATq6kbzyZVT/Fel2soOKeqvJuXE5vtmHF1UWAiAuegsDmo4nsk0bVRZjbcV4rQS2mqMBxH66VaL2ytFd6qOzVev0NUehoYFSYOcZJAXuwqXRL1zKTwvut9v1PyPbLBumU1676Y6qHkSYVx4MsmxqP5lNbEu96RQ4InoC52RnD5dk963xOXU0KQFxRwQJNzvGClR1LQPOBLGQ2bGBQVa2kS0jmzKK7X6Qm9zZBsGEq0TTzFFwrY8+5g/76gZ3AfVw0ARzH9bwd7vQMHvaQgVEa3GMgjL6tah9qeYFfXxOpUX58htgYbKrXp8RTkmAEP+Cbscr9DApW1lNFAIt/4Ut7m/COVM79DBqQk5rdDjgfkFNPM4YEkqZUx0AsY/ZeCENg6oKGRBuGY5CrcU6Hsf8jSJEYdckvQgkcYqIM6WpZznNyr4AQssldBSmjpQ2Z9J6ePQ1q4HkrgBkoh60KwR4SA1noPisHiFQloX0sJCeFzul1SPphrm616E4z/M6yp8qbbs+pSBx9VdFTQQKemAgEQ7yC3FKmC7dgxSe8fug1dbBgXiTirjiUQe6UZCi8ioAAH1sVXcopt1aQLGLrmZUPXsHfQe0Ay7z385D4/9w8PPoEj98P7PRvLjpPgwGtC18EtkbXXe1jC1HG0ArUnY2AsiE7xQ1LEta0nvJpcKhgM5CVtCi2Lyedl3vTIe5k0cCt0lmrCalnXMJkVi7yt23PfEi9tbmYGzUP789J1GYPJfkmzWRmTxUgcdodwb0TEzEnVBATZDZYjJ04hzUcH9KKYGZurKUopEvQrkXncOfB3uJf24f7IVoqw1ci6O+B8bNy3UNQRBDQ5QuAypT3hdPInAiSpWSHMykyjpsovez2NgcIYRLIxCNWXOdq1ft7Z82Haj8T7lfD25e1ICTn9nh4pnZyEAVdLZvG2Ve5oFXM3gc79dMHAiA353vRNMHyOI0oigWTDvpx0hIoEyXi618vTXJ3FkHB4u02HqZ5zs59/NoFPO5yrzhzvF1r6Vkb2SIN9wCyPUiRsFEmpeIQoJ0Md7qwiDSkCHSJ5CWt4SlHtYP+Tw09aHWSJpdUOAwC0yEjfydaTt4g2C34JiQQpcmZbFReMDkSvvohMOkpDtzYtkr31p5NsVwEM9HNGSUsM2t6pZF+k0778hL4YjvpjRAFE4VJyZ5z2u/rmZH02Zjq3mDk1brJlaOWBILCtUmVfn41TADdm6uZBstboBfsGMRWpPoxubn1vPSc8vOaI2aM5Wsw/UrHxsoYxRgzR0iS0lYyX4uQJIprZRBsUcVhbkn2mEgM89O2A6ZjkGkRnOLxzHDzA47uOxpq+11m+XCUt7IxsZGYcTO1Ixww/a87fz2RuFg+S0y2dVQxs0oOkORWNdQoQ8gqWqWsh/iVPo8iiEBCj6PfisqW/LXJhpGTJbn5WIZ/Hyxb4Q+9BlwzhocOmDCLAYc/cPYKPzSag3goyEyXqsxZvBf1DiLBEavBsosCTDWbm5gj/FmmmLZlM09PBZtD9YErJM9/ZQMzxbMu7k37x5RU2jtgsIST1is5vtRx+7jjdCeOdLbIsfoGgKVh0TMCU5hDWzFrqV4a9xdqXtQg8OXHPi4TB0WsnpkmSGpHFgEYRJ8I11ct9vFqLDDVl3BFyN1CNL/eomXjURR665d6A19kxkVnasdcvEpvlO4eGj8tVaFCJtkwI0y7S+0p6zO1i6z3SdwIn2cs2RS5TTYCKF5o77DNrupKuTHYWhmTobUP1a+B4H5oX36UNl2k6M7lSao3L0bw3X75jUP1CEqWFyHCWmdhcSaa62sUNbeJHLF70PCdIbhZNT2YGKklpYZTDbG/lqrlJnuQsv8HChnpplVuEe+ksSDpTM0WJjfCareGcEYmvukwt47M7tmkT8ptC1nwDs2mSBqY4hBh6D4zXb5yM6hpDXA/ZcidMJqtVDQjQ3hyhToWIQYmFGaG14t2ahzspO+NDYaKtinuqtsiQJGr6oL/VNAuIHFbpivKcNkEAjaRcEhPz7IOiQapELdVQP/e/MgR1IV60l15wYT0vcDNcJJ5LZkiX9ApXksR8wJXcJObVtEAapwdSdzNvb9TmYFuUWPp8bhCZH/e7B7PLznJzz6k0XYqfJhjhchMdC4PDhWQLiNWLMjt8q8yPLA9tYcj9zsH8Uh5m+ixHRvEm9u3BotA+c4EMCiZdnUIbXV110CKgdwjp3lwTp5/O41v0PvfD1XaxgO/dIBlcHL287E8XA14KEK1dd9W88dO5VEqnTPx58e1l/+LLy/6c04abHOJjGrfHIS9IgcqIOxz4/jShcrI9RfyVrdGeI4n4BS5NvGyLiJoMem2zM77dnt7b724adgp/L9TO4VlCqZYAAOB2aGoa2Xe8CN9i/ad2FE0NLcONZC3oW4kwfxDigRxw1sZD0e/Js6ZWpn3ZEuFSer6P+2vYDm2tkWS0AA98y+sV5PE0kUnpHV7TjuycYTiVwhP9Cbb5Rn6HJDCG7x+JiZgbLFM9sGWStEM55d0bkybUNX1TucLBMB2roefPEn3pRlVdrHwGbb4c4XmY1xgdw4bkqVHEyaZz3Trz5TWv/PSdI46cbuQNDzcYGLNdz3C16MDfZvO9B3ramZfjCNUrVMdS9XqUowzyUzB8Y5cJNDZFFuVdyaH83Xe7yKNM1r373crA0iD+3GrVq9iCtHuhTncazousGLLMdGzNqYf2plp9o40t16fzhcq7/h1SkF4LYeWIUX1O9UrUTrIEB4wKVMOoBR3vWIBNneINM2xmt2oh0R6tMhfXs47lErs8xfXSagps1D0Uf1BB8rwMGVvezuJTZHMJaFU9OOzyH14u6y8Pzau0/kqaLEnh/sVJv7s/AytZms9UPtsXMEeRidtUsJHc3UPIjVdxSYtTkF5xsL4F7dEDpleixOPCP9n6So8a01mXXZyPbeaDdv2F9fNn0Hw95XWeVUYTM7k/YIEKGwI4qyF67jIBPt9cfre6cGY4REaaPdzq6z6btFho19u12bd2z3ZHrgl4lyZgJrP5bsUllWeJxJ2Ef/tq/Ltnnz/5hioeX7bYtshMp/p5Tz/PoslSH7zRB+FpLPlj1NvzmXMbYi5Ai5mI6V2MwDbbdS9NLC5ktPdpUtJmon8NGDFJ3SmhIPCMe2T5nkbxurrOu5T2R1erRv80kdfkm8pQKh+xZDHZ7jMNMCi3YW6GNBbOYX5T5KhcTO4b0TLmlMxrJOVAHPgi/+lO5jRpXd14oUxK8g/ymbGJNcWeXnPHBFc0w4qXx/bb7LxtB41ci5G4NKuIqH+pyFJneotAaTz3snQNMEk4dW2QuYhKPdRejtqWXDEKqyIc/R00OwVaZTVlgaeBHVrdJ9MM63qPNcfzWtiFi9IWbIErHnjElbohfVZqJO5vAbu1WorJ3csNlS2xuPa6zd6TBK1kIq0aW1uaZ69mZ9Vd2VpSnyM16L//5qgM4DtjI9MMFNTpcyIa/32ktkPvb3DA+xRsmPnt3OvHwzwaM2c6Z2S4njo0cHwY8zsOvfhVdk7AfL/tipOtGXdu452eSRBO23mV23inJ/UYdNdllIUt6nhbt2Nb/uy3+B+seX0+CLR3mPklkOlMQLnDxssz/NivUxiXRzP1FK9LypUXQka16b6FwBH77dKKFgFxpfqZRr9spH4rZO2fKbVbWb2pHyMFIURAWY9Iw/H5eL4UmQihTvsqIkvKYlHYDFyMPHvxpFXaYsUCUVn9H/Oi9VYlPNcyI5wx35SxP9lK1k47B1ZPhJHNAtIkKbG+u4olejtnfG9e6DZRZjkcpdTIzs6QM0m0XqOh/HUjHbmhNOW4DUPRRdxGEV5Rq8gMk+h7Tqd/MquiegkEnUZOlhaicbi+CHKmjuaglZCFWc5RUFobzTenFnPzBts2PSrcFVMKG3V4oYnQ0itRVwyesTcKivjkZCsqtG/mrveBbk3o4NgkzTfqxLp2zniQoT8TZhIYa0nJ/7si9pW8wdXBgxGhTTGA7r0PNoGEtqZhdKO2+akRd+U8JKVouJ3ipnWNqx/oj+NE3SFyPIqHu3uxugSzVqJbyiXcLHTH5W2wUrBKIY+0wuK4QyYSLI7zyC2HPstAF4z6oNstORAdiN19kbP9u/wuu0Ordf0WJ455n+x3ZVnb3eSu0wzTIxyq5AOtQxkqiz0HKuvoaoBC8x8Gk7Q7i6xIINOn4cGhnW4nk8+ed9ApTBuSG1iSlxDqfhlW0yYDXM3PUmuh3HSmu3hDpDTvxHKk3uLic3WxPC6SunYlP85xnoCuiBhoXXsSqkcWlBMsl29QR4eunA8x2eRE05STLzvl4ppZ7Zqr6CBwZioaWxG61UTwcOIpZOzWyCKK5o2tHA6oaFTUtO0FXEjUSv6B80Rb+AEVxYtd4PZErq5C+orp+pR39v2ZBws4WMn19CiuJ3bQUUrJ6BqT505VcGQFiozpib47At8++tLskuKsNI3iRZCSIOnyIP4I+8ZzzHl/Un4vbZ1yk2wQE4646x9RUKA6YjCJp1v87xeMJNJBiRlCgv4l3mjsPMaMOdfRFzSLLrFJ+m1qZpwCw3oTSiKcCr3mtiZE3VJmMhuEUWCeztt1gWXn/Ya4nyTQ5E7NRxQnjkSe0CUGI6Y4gWjmax6XcEY5tvkNKsemsVKgJIrqV0QgtmSyBUJqwWeTHGSQWJKCmjyayyKQlriN/OMCUROB0Kj7HsexGpROG0DCKN9rGqQWJX+QN543SitV+cdpV9GaLTfj07IFR8hAdBOWASLBjBzifwrROZrB98jEt7EAG7PCt5fnkMOlHasUpkBsiRy02L5ZL32LvhoV1w+qsOdqesWkLoF41bwCVpOyntC4uxB0SAmvS54HK8dDc+IQungGrMC6UMsVMtd0TxQJqRKrJgvjtZONxWye5h56bI169Y4mA4DabJ0Zr8I6Wb6uE/TGCGHMUVUWaQmoSUkAZ/e1STKwrmWqMsd6d2nnKBLvG8FPNXAmET8pN06ZusW2RiubTVs9l5nzF0OftFQLU+l9scibmRQJ3subKxO6d8R6Z9vqqKVP2pM1LfCUclyXhPZS9vhS4/2m+IeJlQ4IHicKvf5moN1eEw3RMcePGg/LQctvRJDwnhvUPOqK3Rweb49Z2iqjw3tUr0hse6sh9JYDuPbUU+oMJTtGL5bdJqLWqMr0aKx0VlaLVRLU1YWjd0qgkkJvJE5rVgO0f9+ogFQj1T2og13odIwyp6rFUU2OE4Bx9K3JnzI9Ux0OgzGOvlWtziPV+ORlv5TYl/ui1nFKf6l6nhuw+BdT8txeyV9E0TricreXrx4rNjdTskaf2TATDa8pwl2uj9J7T7yK0kkmV5bAHVeyW+axIPWNvziOVe8SedmRlzj3TqH+d1ZnfgDhozJDkPw+uhKuWQVNwBdAaKgYJ3//IFcG2/50RHCcviF07RZLetq7Zh1bdOg58ObFfO0wpHLbBFzlD8wp0YRfBV+WX1qlsLy9ecfW7UW2P0WHiw0jwHRkZza2HhNuUtZZu/6cbl6dQkmiyebgVUd5A0v0vmRX71ZTPOot5p7uat7F6vLuxWpyCTAv9MKDFcfEJ7/ixQ5zbAcnFdt0bpys0R97hzd5vVtjvKRssDJZECpDny5wPRSVS/nzs5CZ2R/lMrFqn3Zus105O06Rf6z0D05Zs3itgJxmjnOPnpokAmdsI3D+ysaCOYaJyTrRoG6T8Okiczb6sbPRF5eLSaZTJ815/kbbquSar5kijehTV4aubyZ8ji+P9LO+1/AtdwhpqrnIhsfvUxBIHoHatNdzEoyqTdsrIBeVxF7HOFNJGj4bnDq0vdhb8YZ9o82RxJ4L4Sl3kb9nNOwfjuom3uni/Zhs7CxtSFP6QZhlQEAZrUJZ9Mlx0K58AAylCBho57m4wPxNOgKEEJRPPniAFXHq29AEJw7CiYDAA0PyNvmzzqOmDxvl/mxkfeEEvLie2fbNexNEzzRlsm6f74rcXHnjxa05Q3jrjEL4fC4HF++Gu+HuSFyb+vwxMT/asGFkGzgDvXh1tHf06vHLV/jS9puZKjyL5uEsGmQwf/RnA8pDbfsEEwz37sGFSRM4080dEVbSpRQ+GIawo48c0UHR4EDsFuCR3H9k73Th0MQb5ZFc92IeawO2PIMb58bPZLAaXJBoFX+X/dMBsAEvqmvgVSIgzPv3HZj375dwAQQgIHcAeoIOhqFmFzL1isnFejtjuNbgGcOo+q9eDPb2+89fPh700GvZI6vS5o3+WZghXpMx2B+ejoanb0aPypWHPU7yzS2crhcgJ8bF3R+mFr1i2FfOU9EAiQltfp6zZGMZKGjQVlWPN+PUv1jLEvT3erlOyVR0GIWZokfD/Zw0u30XiY3k+EH+bDXinftm/BcvO/ptbAC7bFfo+fA68tATObxNOgV14Ptw2S3Ph5kTPOZUMS4AQJpb4iAoDLITfIB6dPH01ZZ42A8zxr0g/1k/gdM4cDA4uN8dLhvbBhWkI7wLl/E54N5w5Ad9jROEdA98Wt/5wRzetVkbj5eQbJKBfKLjBw+HIcaXE0JJKFTF4yOCr46P8erNI+3U7XH/p/WYq5bCMoMVgQBuLqH/M/zMPjZ0KXolyRMn8cILutV7c8eL0QBznbuZcQgtqLLRy/BGp0QZYpMn2k90oIutHsn9EbZrvnGF6+M1Y/tw57lChDIAmK652/WvT6wIEwJ9/U8RHltLofjNkfgzX5NAscm3DquG61YuXuEf0/NxvsfmstJOOenP3P9c750yaaX63oVU2JXZ5SbYm19C4x9cWA3YcFe0VnLnmE3wWVuFF8VllZJ2+Zh88ny4K5LYbklIQzN1i1LLuaVbG6hf7KK5RNw0XW8o3J9bQVCUSc4Vj/XkxudW1HLSLCEogvsqfCmMg5eBaOTrEJrMUPn98eWkSsW+7XkXGDwO9UDoYWd/fpniSe4IJ8n5Ly/Qs709z15nDC84s4/rvWaYbd5SJSk4x2PxSRrjRgaEW4yN9G+ufcTdIZAIzD17IVWN9knncXK8ZTznC/6yaq8QEa6z2Tg076z7Fy9cqL5RuvOvqbe3JxhletrFZuD/uyNc2CcXBWHOsPbWiALjBfAI2MDdbvbW39YNrelO1AjJHlkCleImx8DAv74dSfNXLtrYItP/NbVIDCJqmThj+dTg7SVTCRpXgcA1Zg9C0XZYU76mGcRDk26wlGwQj/ml5d6LlAflBWIvCDt6X7xoZuW7SS5l7mRGlH3YcUx/cmHJ4KY7mUyAh63FnDc7xRm8xk8scOIsYSrl7hLIaNKpukkASv1WHKhLjrTWhCrCG7XItokWog71UJFYF2+Fyf/mb/yip9bO/wNGzZLvyKMAAA==")))
open("/content/montage/logo_locapay.png", "wb").write(base64.b64decode("..."))
sys.path.insert(0, "/content/montage")
import importlib, montage_auto as M; importlib.reload(M)
import torch
print("✅ Installation terminée —", "GPU actif 🚀" if torch.cuda.is_available() else "⚠️ pas de GPU (ça marchera, mais plus lentement)")

In [ ]:
#@title 2️⃣ Réglages
#@markdown **Couleurs des écrans de couleur et de l'écran de fin**
couleur_1 = "#2E2C51" #@param {type:"string"}
couleur_2 = "#0AC2A4" #@param {type:"string"}
ecrans_de_couleur = 2 #@param {type:"slider", min:0, max:4, step:1}
#@markdown **Sous-titres et découpe**
position_sous_titres = 0.66 #@param {type:"slider", min:0.40, max:0.85, step:0.01}
couper_les_hesitations = True #@param {type:"boolean"}
pause_max_conservee = 0.45 #@param {type:"slider", min:0.20, max:1.00, step:0.05}
#@markdown **Son**
effets_sonores = "léger" #@param ["léger", "aucun"]
ajouter_une_musique = False #@param {type:"boolean"}
volume_musique = 0.18 #@param {type:"slider", min:0.05, max:0.50, step:0.01}
#@markdown **Logo** (cochez « autre_logo » pour charger un PNG à la place du logo Locapay)
logo = True #@param {type:"boolean"}
autre_logo = False #@param {type:"boolean"}
#@markdown **Transcription** — mots rares à bien reconnaître, séparés par des virgules
vocabulaire = "Locapay, démarcheurs, loyer, propriétaire, locataire, Cotonou, Bénin" #@param {type:"string"}
modele = "turbo" #@param ["turbo", "large-v3", "medium", "small"]
#@markdown **Vidéo déjà sur Google Drive ?** Indiquez son chemin (ex. `MyDrive/Videos/rush.mp4`), sinon laissez vide.
video_sur_drive = "" #@param {type:"string"}

R = M.Reglages(couleur_1=couleur_1, couleur_2=couleur_2, nb_ecrans_couleur=ecrans_de_couleur,
               position_sous_titres=position_sous_titres, supprimer_hesitations=couper_les_hesitations,
               pause_max=pause_max_conservee, effets_sonores=effets_sonores, volume_musique=volume_musique,
               logo_en_coin=logo, fichier_logo="/content/montage/logo_locapay.png" if logo else "",
               police_serif="/content/montage/polices/serif.ttf", police_grasse="/content/montage/polices/gras.ttf")
print("✅ Réglages enregistrés")

In [ ]:
#@title 3️⃣ Charger la vidéo (et la musique)
from google.colab import files
import shutil
os.makedirs("/content/montage/entree", exist_ok=True)
EXT_V = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".3gp")
if video_sur_drive.strip():
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = os.path.join("/content/drive", video_sur_drive.strip().lstrip("/"))
    assert os.path.exists(VIDEO), f"Fichier introuvable : {VIDEO}"
else:
    print("👉 Sélectionnez la vidéo brute :")
    up = files.upload()
    vids = [n for n in up if n.lower().endswith(EXT_V)]
    assert vids, "Aucune vidéo reconnue (formats : mp4, mov, m4v, avi, mkv, webm)."
    VIDEO = f"/content/montage/entree/{vids[0]}"
    shutil.move(vids[0], VIDEO)
MUSIQUE = None
if ajouter_une_musique:
    print("👉 Sélectionnez la musique (mp3, wav, m4a…) :")
    up = files.upload()
    if up:
        n = list(up)[0]; MUSIQUE = f"/content/montage/entree/{n}"; shutil.move(n, MUSIQUE)
if logo and autre_logo:
    print("👉 Sélectionnez votre logo (PNG, fond transparent de préférence) :")
    up = files.upload()
    if up:
        n = list(up)[0]; R.fichier_logo = f"/content/montage/entree/{n}"; shutil.move(n, R.fichier_logo)
DUREE = M.duree_video(VIDEO)
print(f"✅ Vidéo : {os.path.basename(VIDEO)} ({DUREE:.1f} s)" + (f" — musique : {os.path.basename(MUSIQUE)}" if MUSIQUE else ""))

In [ ]:
#@title 4️⃣ Transcription automatique
mots_bruts = M.transcrire(VIDEO, modele=modele, langue="fr", vocabulaire=vocabulaire)
print("\n📝 Transcription :\n")
print(M.texte_transcription(mots_bruts))

In [ ]:
#@title 5️⃣ Corrections (facultatif) et plan de montage
#@markdown Format : `mot_mal_reconnu=bon_mot; autre=correction` — ex. `Locapé=Locapay; démarcheur=démarcheurs`
corrections = "Locapé=Locapay; Lokapay=Locapay; Locapaï=Locapay" #@param {type:"string"}
mots = M.appliquer_corrections(mots_bruts, corrections)
PLAN = M.construire_plan(mots, DUREE, R)
print(M.resume_plan(PLAN))
print("\nLes mots en MAJUSCULES seront en gras. Relancez cette étape après une correction.")

In [ ]:
#@title 6️⃣ Montage et téléchargement
from IPython.display import Video, display
os.makedirs("/content/montage/sortie", exist_ok=True)
base = os.path.splitext(os.path.basename(VIDEO))[0]
SORTIE = f"/content/montage/sortie/{base}_monte.mp4"
print("🎬 Montage en cours…")
M.monter(VIDEO, PLAN, R, SORTIE, musique=MUSIQUE)
M.ecrire_srt(PLAN, SORTIE.replace(".mp4", ".srt"))
open(SORTIE.replace(".mp4", "_transcription.txt"), "w", encoding="utf-8").write(M.texte_transcription(mots))
print("✅ Terminé !")
display(Video(SORTIE, embed=True, width=320))
files.download(SORTIE)

---
**Fichiers produits** (dossier `/content/montage/sortie`, panneau 📁 à gauche) : la vidéo montée, les sous-titres `.srt` et la transcription `.txt`.

**Astuces**
- Filmez en vertical, la personne au centre, avec un son propre : la transcription n'en sera que meilleure.
- Trop de coupes ? Montez « pause_max_conservee » (étape 2). Pas assez ? Baissez-la.
- Sous-titres qui cachent les mains ou le visage ? Ajustez « position_sous_titres ».